In [11]:
import importlib
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))

# Принудительная перезагрузка модуля
if "scripts.db_config" in sys.modules:
    del sys.modules["scripts.db_config"]

import scripts.db_config
importlib.reload(scripts.db_config)

from scripts.db_config import read_sql, execute_sql, execute_many

# ДИАГНОСТИКА
import inspect
print("Файл модуля:", scripts.db_config.__file__)
print("read_sql signature:", inspect.signature(read_sql))
print("execute_sql signature:", inspect.signature(execute_sql))
print("execute_many signature:", inspect.signature(execute_many))

Файл модуля: d:\Projects\abc_xyz\scripts\db_config.py
read_sql signature: (query: str, params: Any = None) -> pandas.core.frame.DataFrame
execute_sql signature: (query: str, params: Any = None) -> None
execute_many signature: (statements: list[str]) -> None


In [12]:
# Размер таблицы
size_df = read_sql("""
    SELECT
        COUNT(*)                AS total_rows,
        COUNT(DISTINCT invoice_no) AS unique_invoices,
        COUNT(DISTINCT stock_code) AS unique_stocks,
        COUNT(DISTINCT customer_id) AS unique_customers,
        MIN(invoice_date)       AS min_date,
        MAX(invoice_date)       AS max_date
    FROM raw_transactions;
""")
print("Общая информация о raw_transactions:")
display(size_df.T)

Общая информация о raw_transactions:


,0
total_rows,1067371
unique_invoices,53628
unique_stocks,5305
unique_customers,5942
min_date,2009-12-01 07:45:00
max_date,2011-12-09 12:50:00


In [13]:
missing_df = read_sql("""
    SELECT
        COUNT(*) FILTER (WHERE invoice_no   IS NULL OR invoice_no   = '') AS inv_missing,
        COUNT(*) FILTER (WHERE stock_code   IS NULL OR stock_code   = '') AS stock_missing,
        COUNT(*) FILTER (WHERE description  IS NULL OR description  = '') AS desc_missing,
        COUNT(*) FILTER (WHERE quantity     IS NULL) AS qty_missing,
        COUNT(*) FILTER (WHERE invoice_date IS NULL) AS date_missing,
        COUNT(*) FILTER (WHERE unit_price   IS NULL) AS price_missing,
        COUNT(*) FILTER (WHERE customer_id  IS NULL) AS cust_missing,
        COUNT(*) FILTER (WHERE country      IS NULL OR country      = '') AS country_missing
    FROM raw_transactions;
""")
print("Пропуски по колонкам:")
display(missing_df.T.rename(columns={0: "missing_count"}))

Пропуски по колонкам:


,missing_count
inv_missing,0
stock_missing,0
desc_missing,0
qty_missing,0
date_missing,0
price_missing,0
cust_missing,243007
country_missing,0


In [14]:
dup_df = read_sql("""
    WITH dup AS (
        SELECT
            invoice_no, stock_code, quantity,
            invoice_date, unit_price, customer_id,
            COUNT(*) AS cnt
        FROM raw_transactions
        GROUP BY invoice_no, stock_code, quantity,
                 invoice_date, unit_price, customer_id
        HAVING COUNT(*) > 1
    )
    SELECT
        COUNT(*)         AS dup_groups,
        SUM(cnt - 1)     AS total_extra_rows
    FROM dup;
""")
print("Дубликаты (по всем 6 полям, кроме description и country):")
display(dup_df.T)

Дубликаты (по всем 6 полям, кроме description и country):


,0
dup_groups,32909
total_extra_rows,34337


In [15]:
cancel_df = read_sql("""
    SELECT
        COUNT(*) FILTER (WHERE invoice_no LIKE 'C%') AS cancelled_rows,
        COUNT(*) FILTER (WHERE invoice_no LIKE 'C%' AND quantity > 0) AS cancel_positive_qty,
        COUNT(*) FILTER (WHERE invoice_no LIKE 'C%' AND quantity < 0) AS cancel_negative_qty,
        COUNT(DISTINCT invoice_no) FILTER (WHERE invoice_no LIKE 'C%') AS cancelled_invoices
    FROM raw_transactions;
""")
print("Отмены:")
display(cancel_df.T)

Отмены:


,0
cancelled_rows,19494
cancel_positive_qty,1
cancel_negative_qty,19493
cancelled_invoices,8292


In [16]:
neg_qty_df = read_sql("""
    SELECT
        COUNT(*) AS negative_qty_rows,
        COUNT(*) FILTER (WHERE invoice_no NOT LIKE 'C%') AS negative_qty_non_cancel,
        COUNT(*) FILTER (WHERE invoice_no LIKE 'C%')     AS negative_qty_cancel
    FROM raw_transactions
    WHERE quantity < 0;
""")
print("Отрицательный Quantity:")
display(neg_qty_df.T)

print("\nПримеры отрицательных Quantity вне отмен:")
display(read_sql("""
    SELECT invoice_no, stock_code, description, quantity, unit_price, customer_id
    FROM raw_transactions
    WHERE quantity < 0 AND invoice_no NOT LIKE 'C%'
    LIMIT 20;
"""))

Отрицательный Quantity:


,0
negative_qty_rows,22950
negative_qty_non_cancel,3457
negative_qty_cancel,19493



Примеры отрицательных Quantity вне отмен:


,invoice_no,stock_code,description,quantity,unit_price,customer_id
0,491046,20987,nan,-100,0.0,None
1,491047,21982,nan,-50,0.0,None
2,491065,84855,MIA,-108,0.0,None
3,491066,35949,nan,-67,0.0,None
4,491108,84922,nan,-360,0.0,None
5,491128,21490,nan,-20,0.0,None
6,491137,21489,nan,-8,0.0,None
7,491141,22174,nan,-58,0.0,None
8,491167,17013A,nan,-45,0.0,None
9,491206,84841,nan,-54,0.0,None


In [17]:
price_df = read_sql("""
    SELECT
        COUNT(*) FILTER (WHERE unit_price = 0) AS zero_price_rows,
        COUNT(*) FILTER (WHERE unit_price < 0) AS negative_price_rows,
        COUNT(DISTINCT stock_code) FILTER (WHERE unit_price <= 0) AS stocks_with_zero_price
    FROM raw_transactions;
""")
print("UnitPrice <= 0:")
display(price_df.T)

# Посмотрим примеры нулевых цен
print("\nПримеры строк с unit_price = 0:")
display(read_sql("""
    SELECT invoice_no, stock_code, description, quantity, unit_price
    FROM raw_transactions
    WHERE unit_price = 0
    LIMIT 20;
"""))

UnitPrice <= 0:


,0
zero_price_rows,6202
negative_price_rows,5
stocks_with_zero_price,2972



Примеры строк с unit_price = 0:


,invoice_no,stock_code,description,quantity,unit_price
0,491971,85042,ANTIQUE LILY FAIRY LIGHTS,1,0.0
1,491972,21100,bad quality,-76,0.0
2,491974,72752B,nan,-21,0.0
3,491975,72752C,nan,-24,0.0
4,491973,72752A,nan,-18,0.0
5,491978,72755C,nan,-60,0.0
6,491977,72755B,nan,-23,0.0
7,491976,72752D,nan,-46,0.0
8,491990,C2,nan,100,0.0
9,492000,21719,nan,-81,0.0


In [18]:
# Известные служебные коды (не товары)
special_codes = (
    'POST', 'D', 'M', 'BANK CHARGES', 'DOT', 'S', 'AMAZONFEE',
    'B', 'CRUK', 'PADS', 'C2'
)

# Используем positional-параметры: %s с tuple
special_df = read_sql(
    """
    SELECT stock_code, description, COUNT(*) AS cnt
    FROM raw_transactions
    WHERE stock_code = ANY(%(codes)s::text[])
    GROUP BY stock_code, description
    ORDER BY cnt DESC;
    """,
    params={"codes": list(special_codes)},
)
print("Служебные StockCode в данных:")
display(special_df)

Служебные StockCode в данных:


,stock_code,description,cnt
0,POST,POSTAGE,2115
1,DOT,DOTCOM POSTAGE,1444
2,M,Manual,1421
3,C2,CARRIAGE,279
4,D,Discount,177
5,S,SAMPLES,104
6,BANK CHARGES,Bank Charges,96
7,AMAZONFEE,AMAZON FEE,43
8,PADS,PADS TO MATCH ALL CUSHIONS,19
9,CRUK,CRUK Commission,16


In [19]:
date_range = read_sql("""
    SELECT
        MIN(invoice_date) AS min_date,
        MAX(invoice_date) AS max_date,
        COUNT(DISTINCT DATE_TRUNC('month', invoice_date)) AS total_months
    FROM raw_transactions;
""")
print("Диапазон дат:")
display(date_range.T)

# Как распределяются транзакции по годам
yearly = read_sql("""
    SELECT
        EXTRACT(YEAR FROM invoice_date)::int AS year,
        COUNT(*)                              AS rows,
        COUNT(DISTINCT invoice_no)            AS invoices,
        ROUND(SUM(quantity * unit_price)::numeric, 2) AS revenue
    FROM raw_transactions
    WHERE quantity > 0 AND unit_price > 0 AND invoice_no NOT LIKE 'C%'
    GROUP BY 1 ORDER BY 1;
""")
print("\nРаспределение по годам:")
display(yearly)

Диапазон дат:


,0
min_date,2009-12-01 07:45:00
max_date,2011-12-09 12:50:00
total_months,25



Распределение по годам:


,year,rows,invoices,revenue
0,2009,43957,1682,825685.76
1,2010,509088,19994,10303952.40
2,2011,488625,18401,9842956.40


In [20]:
"""
Пересборка sales с безопасным фильтром.

ПРИНЦИП: фильтруем ТОЛЬКО по конкретным StockCode (NOT IN).
НИКАКИХ regex по описанию.
Паттерн sales_new + RENAME — не блокирует читателей.
"""
from scripts.db_config import execute_sql, execute_many

SPECIAL_CODES = (
    'POST', 'D', 'M', 'BANK CHARGES', 'DOT', 'S', 'AMAZONFEE',
    'B', 'CRUK', 'PADS', 'C2', 'C3',
    'ADJUST', 'ADJUST2',
    '23444', '23574', '22016',
)
codes_sql = ", ".join(f"'{c}'" for c in SPECIAL_CODES)

# 0. Убиваем idle-in-transaction (защита от блокировок)
execute_sql("""
DO $$
BEGIN
    PERFORM pg_terminate_backend(pid)
    FROM pg_stat_activity
    WHERE datname = current_database()
      AND pid != pg_backend_pid()
      AND state = 'idle in transaction';
END $$;
""")

# 1. Удаляем возможный sales_new с прошлых попыток
execute_sql("DROP TABLE IF EXISTS sales_new;")

# 2. Создаём sales_new
execute_sql(f"""
CREATE TABLE sales_new AS
SELECT
    invoice_no, stock_code, description, quantity, invoice_date,
    unit_price, customer_id, country,
    quantity * unit_price                   AS revenue,
    DATE_TRUNC('month', invoice_date)::date AS month_start,
    EXTRACT(YEAR  FROM invoice_date)::int   AS year,
    EXTRACT(MONTH FROM invoice_date)::int   AS month
FROM raw_transactions
WHERE
    invoice_no NOT LIKE 'C%'
    AND quantity > 0
    AND unit_price > 0
    AND stock_code NOT IN ({codes_sql})
    AND stock_code NOT LIKE 'gift_%'
    AND (description IS NULL OR description NOT LIKE 'Adjust bad debt%');
""")

# 3. Индексы
execute_many([
    "CREATE INDEX idx_sales_new_stock    ON sales_new (stock_code);",
    "CREATE INDEX idx_sales_new_date     ON sales_new (invoice_date);",
    "CREATE INDEX idx_sales_new_customer ON sales_new (customer_id);",
    "CREATE INDEX idx_sales_new_month    ON sales_new (month_start);",
])

# 4. Подмена: DROP старой + RENAME
execute_sql("DROP TABLE IF EXISTS sales;")
execute_sql("ALTER TABLE sales_new RENAME TO sales;")

# 5. Переименовываем индексы
execute_many([
    "ALTER INDEX IF EXISTS idx_sales_new_stock    RENAME TO idx_sales_stock;",
    "ALTER INDEX IF EXISTS idx_sales_new_date     RENAME TO idx_sales_date;",
    "ALTER INDEX IF EXISTS idx_sales_new_customer RENAME TO idx_sales_customer;",
    "ALTER INDEX IF EXISTS idx_sales_new_month    RENAME TO idx_sales_month;",
])

print("✅ sales пересобрана через sales_new + RENAME.")

✅ sales пересобрана через sales_new + RENAME.


In [21]:
check = read_sql("""
    SELECT
        (SELECT COUNT(*) FROM raw_transactions) AS raw_rows,
        (SELECT COUNT(*) FROM sales)            AS sales_rows,
        (SELECT COUNT(DISTINCT stock_code) FROM sales) AS unique_stocks,
        (SELECT COUNT(DISTINCT invoice_no) FROM sales) AS unique_invoices,
        (SELECT COUNT(DISTINCT customer_id) FROM sales) AS unique_customers,
        (SELECT MIN(invoice_date) FROM sales) AS min_date,
        (SELECT MAX(invoice_date) FROM sales) AS max_date,
        (SELECT ROUND(SUM(revenue)::numeric, 2) FROM sales) AS total_revenue;
""")
print("Сравнение raw vs sales:")
display(check.T)

Сравнение raw vs sales:


,0
raw_rows,1067371
sales_rows,1036925
unique_stocks,4895
unique_invoices,39526
unique_customers,5852
min_date,2009-12-01 07:45:00
max_date,2011-12-09 12:50:00
total_revenue,20109033.72


In [22]:
check = read_sql("""
    SELECT
        COUNT(*) FILTER (WHERE quantity <= 0)        AS non_pos_qty,
        COUNT(*) FILTER (WHERE unit_price <= 0)      AS non_pos_price,
        COUNT(*) FILTER (WHERE invoice_no LIKE 'C%') AS cancelled,
        COUNT(*) FILTER (WHERE stock_code IN (
            'POST','D','M','BANK CHARGES','DOT','S',
            'AMAZONFEE','B','CRUK','PADS','C2'
        ))                                            AS special_codes,
        COUNT(*)                                      AS total_rows
    FROM sales;
""")
print("Проверка sales (все должны быть 0, кроме total_rows):")
display(check.T)

Проверка sales (все должны быть 0, кроме total_rows):


,0
non_pos_qty,0
non_pos_price,0
cancelled,0
special_codes,0
total_rows,1036925


In [23]:
yearly = read_sql("""
    SELECT
        year,
        COUNT(*)                              AS rows,
        COUNT(DISTINCT invoice_no)            AS invoices,
        COUNT(DISTINCT stock_code)            AS unique_skus,
        ROUND(SUM(revenue)::numeric, 2)       AS revenue
    FROM sales
    GROUP BY year
    ORDER BY year;
""")
print("Транзакции и выручка по годам:")
display(yearly)

Транзакции и выручка по годам:


,year,rows,invoices,unique_skus,revenue
0,2009,43829,1671,3047,801216.47
1,2010,506747,19632,4094,9815704.29
2,2011,486349,18223,3811,9492112.96


In [24]:
print("Аномальная отмена (C-счёт с quantity > 0):")
display(read_sql("""
    SELECT * FROM raw_transactions
    WHERE invoice_no LIKE 'C%' AND quantity > 0;
"""))

Аномальная отмена (C-счёт с quantity > 0):


,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country
0,C496350,M,Manual,1,2010-02-01 08:24:00,373.57,None,United Kingdom


## Этап 1. Знакомство с данными и правила очистки — ВЫВОДЫ

### Что сделано
- Загружено 1 067 371 транзакция за период 01.12.2009 – 09.12.2011.
- Изучены пропуски, дубликаты, отмены, отрицательные количества, нулевые цены.
- Выявлены служебные StockCode (POST, DOT, M, C2, D, S, BANK CHARGES, AMAZONFEE, PADS, CRUK, B).
- Сформирована очищенная выборка `sales` — 1 037 053 строк.

### Правила очистки

| Правило | Обоснование |
|---------|-------------|
| Исключены отмены (invoice_no LIKE 'C%') — 19 494 строки | Возвраты не отражают реальный спрос и уменьшают выручку |
| Исключены quantity <= 0 — 22 950 строк | Возвраты, инвентаризация, списания |
| Исключены unit_price <= 0 — 6 207 строк | Бесплатные товары, ошибки, служебные записи |
| Исключены служебные StockCode — ~5 700 строк | Почта, скидки, банк. сборы, ручные правки — это не товары |
| Исключены короткие StockCode (<5) | Некорректные коды |
| Исключены gift-сертификаты | Не физические товары |
| **Оставлены дубликаты** — 34 337 строк | Валидные повторные покупки в одном счёте |
| **Оставлены пропуски customer_id** — 243 007 строк | Для ABC/XYZ не критичны; для RFM отфильтруем отдельно |

### Период анализа
- Базовые метрики, ABC: **весь период** (01.12.2009 – 09.12.2011).
- Временные ряды для XYZ: **24 полных месяца** (2009-12 — 2011-11).
- Декабрь 2011 неполный — исключаем только из XYZ.

### Итоговые метрики `sales`
- Строк: 1 037 053 (−2.84% от исходной).
- Уникальных SKU: 4 899.
- Уникальных заказов: 39 565.
- Уникальных покупателей: 5 861.
- Общая выручка: **£20 120 035.98**.

### Замечания
- Обнаружена 1 аномальная отмена с положительным Quantity — проверена, не влияет.
- Обнаружены 3 457 строк с отрицательным Quantity вне отмен — это корректировки (bad debt, инвентаризация, списания), исключены.
- `PADS` (PADS TO MATCH ALL CUSHIONS) — возможный реальный товар, проверим дополнительно в этапе 3 (ABC).
- `customer_id` отсутствует в ~23% строк — эти строки пригодны для ABC/XYZ, но не для RFM.

### Финальные метрики sales (после итеративной очистки)
- Строк: 1 036 925
- Уникальных SKU: 4 895
- Уникальных заказов: 39 526
- Уникальных покупателей: 5 852
- Продано единиц: 11 402 616
- Выручка: £20 109 033.72

### Финальный список служебных SKU (22 позиции, суммарно £863 561)
M, DOT, POST, AMAZONFEE, C2, B, ADJUST, 23444, ADJUST2,
gift_0001_10/20/30/40/50/70/80, BANK CHARGES, D, S,
23574, 22016, PADS

### Финальный список служебных SKU (22 позиции, суммарно £863 561)
- Ручные правки: M, ADJUST, ADJUST2, B (bad debt)
- Почта: POST, DOT
- Транспорт: C2, 23444
- Комиссии: AMAZONFEE, BANK CHARGES
- Скидки: D
- Образцы: S
- Подарочные сертификаты: gift_0001_10…80, 22016
- Прочее: 23574 (Packing), PADS